# Carga de datos

**Objetivo:** importar la base crediticia y realizar controles iniciales antes del EDA.

Este notebook no modifica el archivo original. Solo lo carga en memoria y comprueba que esté listo para analizar.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Localizar y cargar el archivo

In [ ]:
# Permite ejecutar el notebook desde la raíz del proyecto o desde src/.
posibles_rutas = [Path("Base_de_datos.csv"), Path("../Base_de_datos.csv")]
ruta_datos = next((ruta for ruta in posibles_rutas if ruta.exists()), None)

if ruta_datos is None:
    raise FileNotFoundError("No se encontró Base_de_datos.csv. Revisá la estructura del proyecto.")

df = pd.read_csv(ruta_datos, parse_dates=["fecha_prestamo"])
print(f"Archivo cargado desde: {ruta_datos.resolve()}")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

## 2. Vista inicial

In [ ]:
df.head()

In [ ]:
resumen_columnas = pd.DataFrame({
    "tipo_dato": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=True),
})
resumen_columnas

## 3. Controles básicos de calidad

In [ ]:
print(f"Filas duplicadas exactas: {df.duplicated().sum()}")
print()
print("Distribución de la variable objetivo:")
display(df["Pago_atiempo"].value_counts(dropna=False).rename("cantidad").to_frame())
display((df["Pago_atiempo"].value_counts(normalize=True, dropna=False) * 100).round(2).rename("porcentaje").to_frame())

In [ ]:
valores_objetivo = set(df["Pago_atiempo"].dropna().unique())
assert valores_objetivo.issubset({0, 1}), f"Objetivo inesperado: {valores_objetivo}"
print("Control superado: Pago_atiempo es una variable binaria (0/1).")

## Conclusión de la carga

- La base se cargó correctamente.
- `Pago_atiempo` es la variable objetivo: `1` significa pago a tiempo y `0`, pago fuera de término.
- Los faltantes, valores atípicos y el desbalance de clases se estudian en `comprension_eda.ipynb`.